# Phase 5 Synthesis: Kappa-Scaling, Polariton, Vestigial MC, Categorical Infrastructure, and Gauge Emergence

**Phase 5 Waves 1–2 + 4A–4C — Technical Synthesis Notebook**

**Authors:** John Roehm

---

## Introduction

This notebook synthesizes the Phase 5 results across five waves:

| Wave | Topic | Key Result |
|------|-------|------------|
| 1 | Kappa-scaling predictions | Dispersive $\propto \kappa^2$, dissipative $\propto \kappa$, crossover $\kappa_{\text{cross}}$ |
| 2A | Polariton Tier 1 | $T_H \sim 10^{10} \times$ BEC $T_H$, driven-dissipative corrections |
| 2B | Vestigial MC | Split transition detected in fermion-bag Monte Carlo |
| 4A | Categorical infrastructure | Pivotal, spherical, semisimple, fusion categories in Lean |
| 4B–C | Gauge emergence | Drinfeld double $D(G)$, anyon counting, string-net foundation |

All physics is imported from `src/` modules. No formulas are reimplemented.

## Setup: Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

In [ ]:
import numpy as np
import pandas as pd
import json
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

from src.core.visualizations import (
    COLORS,
    LAYOUT_TEMPLATE,
    FONT,
    TITLE_FONT,
    apply_layout,
    fig_kappa_scaling_physical,
    fig_vestigial_binder_crossing,
    fig_fusion_rules_comparison,
    fig_drinfeld_anyon_spectrum,
)
from src.experimental.kappa_scaling import (
    compute_all_sweeps,
    kappa_scaling_summary,
)
from src.core.constants import (
    EXPERIMENTS,
    FUSION_EXAMPLES,
    CATEGORY_HIERARCHY,
    DRINFELD_DOUBLE,
    POLARITON_PLATFORMS,
)
from src.core.formulas import (
    kappa_scaling_dispersive,
    kappa_scaling_dissipative,
    kappa_scaling_crossover,
    global_dimension_squared,
    frobenius_perron_dim,
    fusion_associativity_check,
    drinfeld_double_dim,
    drinfeld_double_simples,
    drinfeld_double_simples_abelian,
)

## 1. Kappa-Scaling Predictions

The $\kappa$-scaling test exploits different power-law dependences of EFT corrections on the surface gravity:

$$\delta_{\text{disp}}(\kappa) = -\frac{\pi}{6}\left(\frac{\xi\kappa}{c_s}\right)^2 \propto \kappa^2$$

$$\delta_{\text{diss}}(\kappa) = \frac{(\gamma_1 + \gamma_2)\kappa}{c_s^2} \propto \kappa$$

At $\kappa_{\text{cross}} = 6(\gamma_1 + \gamma_2)/(\pi\xi^2)$, the two corrections are equal in magnitude. Below this crossover, dissipative corrections dominate; above, dispersive corrections dominate.

In [ ]:
sweeps = compute_all_sweeps()
summary = kappa_scaling_summary(sweeps)

rows = []
for name, s in summary.items():
    rows.append({
        "Platform": name,
        "\u03ba_nom [s\u207b\u00b9]": f"{s['kappa_nominal']:.2e}",
        "\u03ba_cross [s\u207b\u00b9]": f"{s['kappa_cross']:.2e}",
        "\u03ba_nom/\u03ba_cross": f"{s['kappa_ratio']:.2f}",
        "Regime": s['regime'],
        "\u03b1_disp (expect 2)": f"{s['scaling_exponent_disp']:.3f}",
        "\u03b1_diss (expect 1)": f"{s['scaling_exponent_diss']:.3f}",
        "\u03b4_disp(\u03ba_nom)": f"{s['delta_disp_nominal']:.2e}",
        "\u03b4_diss(\u03ba_nom)": f"{s['delta_diss_nominal']:.2e}",
    })

display(pd.DataFrame(rows))

In [ ]:
# viz-ref: fig_kappa_scaling_physical
fig = fig_kappa_scaling_physical()
fig.show()

The log-log slopes confirm the EFT predictions: dispersive corrections scale as $\kappa^2$ (slope $\approx 2$) while dissipative corrections scale as $\kappa^1$ (slope $\approx 1$). The Heidelberg K-39 platform, with Feshbach-tunable $a_s$, is optimally positioned for this test.

## 2. Polariton Tier 1

Polariton systems offer dramatically enhanced Hawking temperatures ($T_H \sim 10^{10} \times$ BEC $T_H$) due to their extreme surface gravity ($\kappa \sim 0.05$ THz). The Tier 1 perturbative patch applies when $\Gamma_{\text{pol}}/\kappa \ll 1$.

In [ ]:
rows = []
for name, plat in POLARITON_PLATFORMS.items():
    rows.append({
        "Platform": name,
        "\u03c4_cav [ps]": f"{plat['tau_cav'] * 1e12:.0f}",
        "\u0393_pol/\u03ba": f"{plat['Gamma_pol_over_kappa']:.3f}",
        "Tier 1 regime": plat['tier1_regime'],
        "Tier 1 valid": plat['tier1_valid'],
        "T_H [K]": f"{plat['T_H_K']:.4e}",
        "D": f"{plat['D']:.3f}",
    })

display(pd.DataFrame(rows))

The long-lifetime (100 ps) and ultra-long-lifetime (300 ps) cavities are in the perturbative regime ($\Gamma_{\text{pol}}/\kappa < 0.1$), making the SK-EFT perturbative treatment applicable. The standard cavity (3 ps) has $\Gamma_{\text{pol}}/\kappa > 1$, requiring non-perturbative methods.

## 3. Vestigial MC Results

The vestigial Monte Carlo simulation uses fermion-bag algorithm to study the ADW model phase structure. Results are loaded from the latest JSON output.

In [ ]:
# Load the latest MC results
mc_dir = Path("..") / "docs" / "vestigial_mc_results"
mc_files = sorted(mc_dir.glob("vestigial_mc_*.json"))

if mc_files:
    latest_mc = mc_files[-1]
    with open(latest_mc) as f:
        mc_data = json.load(f)

    # Binder crossing analysis
    binder = mc_data.get("binder_crossing", {})
    display(pd.DataFrame([{
        "Lattice sizes": str(binder.get("lattice_sizes", [])),
        "Tetrad crossing": binder.get("tetrad_crossing", "N/A"),
        "Metric crossing": binder.get("metric_crossing", "N/A"),
        "Vestigial detected": binder.get("vestigial_detected", False),
        "Vestigial window": str(binder.get("vestigial_window", "N/A")),
    }]))
else:
    print("No MC result files found.")

In [ ]:
# viz-ref: fig_vestigial_binder_crossing
fig = fig_vestigial_binder_crossing()
fig.show()

## 4. Categorical Infrastructure (Wave 4A–4B)

The categorical foundation for string-net condensation and gauge emergence is built in layers. The hierarchy extends Mathlib's existing monoidal/braided/rigid categories with new pivotal, spherical, semisimple, and fusion category structures.

This is the **first formalization of fusion categories** in any theorem prover.

In [ ]:
# Categorical hierarchy: existing vs new
existing = CATEGORY_HIERARCHY['mathlib_existing']
wave4a = CATEGORY_HIERARCHY['wave4a_new']
wave4b = CATEGORY_HIERARCHY['wave4b_new']

display(pd.DataFrame([
    {"Layer": "Mathlib (existing)", "Count": len(existing), "Structures": ", ".join(existing)},
    {"Layer": "Wave 4A (new)", "Count": len(wave4a), "Structures": ", ".join(wave4a)},
    {"Layer": "Wave 4B (new)", "Count": len(wave4b), "Structures": ", ".join(wave4b)},
]))

In [ ]:
# Concrete fusion category examples
rows = []
for name, ex in FUSION_EXAMPLES.items():
    dims = ex['quantum_dims']
    D_sq = global_dimension_squared(dims)

    # FP dimension and associativity check where fusion rules exist
    fp_str = "N/A"
    assoc_str = "N/A"
    if 'fusion_rules' in ex and ex['fusion_rules']:
        fp_dims = frobenius_perron_dim(ex['fusion_rules'], ex['n_simples'])
        fp_str = ", ".join(f"{d:.4f}" for d in fp_dims)
        assoc = fusion_associativity_check(ex['fusion_rules'], ex['n_simples'])
        assoc_str = str(assoc)

    rows.append({
        "Category": name,
        "Group": ex.get('group', 'N/A'),
        "#Simples": ex['n_simples'],
        "Quantum dims": str([f"{d:.4f}" if isinstance(d, float) else str(d) for d in dims]),
        "D\u00b2": f"{D_sq:.4f}",
        "FP dims": fp_str,
        "Associative": assoc_str,
        "F trivial": ex.get('F_symbols_trivial', 'N/A'),
    })

display(pd.DataFrame(rows))

In [ ]:
# viz-ref: fig_fusion_rules_comparison
fig = fig_fusion_rules_comparison()
fig.show()

All group categories ($\text{Vec}_{\mathbb{Z}_2}$, $\text{Vec}_{\mathbb{Z}_3}$, $\text{Vec}_{S_3}$) have quantum dimension 1 for all simple objects, as expected. $\text{Rep}(S_3)$ has dimensions $\{1, 1, 2\}$ matching the irreducible representation dimensions. The Fibonacci category has the golden ratio $\phi = (1+\sqrt{5})/2 \approx 1.618$ as the quantum dimension of $\tau$.

## 5. Gauge Emergence: Drinfeld Double (Wave 4C)

The Drinfeld double $D(G) = k^G \otimes k[G]$ is the Hopf algebra whose representation category is the Drinfeld center $Z(\text{Vec}_G)$. Its simple modules classify the anyonic excitations of Dijkgraaf-Witten gauge theory with gauge group $G$.

For each conjugacy class $K$ in $G$, the centralizer $C_G(g_K)$ contributes irreps. The total number of anyons is $\sum_K |\text{Irr}(C_G(g_K))|$.

In [ ]:
rows = []
for name, dd in DRINFELD_DOUBLE.items():
    dim_D = drinfeld_double_dim(dd['group_order'])

    if 'irreps_per_class' in dd:
        n_simples = drinfeld_double_simples(
            dd['n_conj_classes'], dd['irreps_per_class']
        )
    else:
        n_simples = drinfeld_double_simples_abelian(dd['group_order'])

    rows.append({
        "Group": name,
        "|G|": dd['group_order'],
        "dim D(G)": dim_D,
        "#Conj classes": dd['n_conj_classes'],
        "#Anyons": n_simples,
        "Expected": dd['n_simples'],
        "Match": n_simples == dd['n_simples'],
    })

display(pd.DataFrame(rows))

In [ ]:
# viz-ref: fig_drinfeld_anyon_spectrum
fig = fig_drinfeld_anyon_spectrum()
fig.show()

For abelian groups ($\mathbb{Z}_2$, $\mathbb{Z}_3$), the number of anyons equals $|G|^2$ since every element is its own conjugacy class and every centralizer is the full group. For non-abelian $S_3$, the 3 conjugacy classes have centralizers of orders 6, 2, 3, contributing 3, 2, 3 irreps respectively, for 8 anyons total.

The Drinfeld center $Z(\text{Vec}_G)$ is always non-chiral (topological central charge $c = 0 \bmod 8$), which is the categorical foundation for gauge erasure: emergent gauge structure from string-nets is always doubled.

## 6. Cross-Wave Summary

| Wave | Lean Theorems | Key Prediction | Experimental Status |
|------|--------------|----------------|--------------------|
| 1: $\kappa$-scaling | $\alpha_{\text{disp}} = 2.0$, $\alpha_{\text{diss}} = 1.0$ verified | $\kappa_{\text{cross}}$ separates regimes | Heidelberg best positioned |
| 2A: Polariton | Tier 1 patch validated for long-cavity | $T_H \sim 10^{10}\times$ BEC | Paris ultra-long cavity feasible |
| 2B: Vestigial MC | Fermion-bag sign-free | Split transition: metric before tetrad | Pilot complete, scaling study needed |
| 4A: Categories | Pivotal, spherical, semisimple | First fusion category formalization | N/A (mathematical infrastructure) |
| 4B–4C: Gauge | Drinfeld double, anyon counting | $Z(\text{Vec}_G)$ always non-chiral | Connects to gauge erasure theorem |